# 0027 / 04 Dataset package and audit

Validate actor leakage, widths, action bounds, and whole-episode split isolation, then write hashed gzip shards for training.


In [ ]:
from __future__ import annotations
import csv, gzip, hashlib, json, re, zipfile
from collections import Counter
from datetime import date
from pathlib import Path
from typing import Any, Mapping

DATA_START = date(2026, 7, 15)
DATA_END = date(2026, 8, 1)
TARGET_DECK_SHA256 = "f50fa3a23cdf21be7cf7d3f558b8ff0b82e8d4e7ba8f61b7b4cacc1a0080c16a"
SCHEMA_VERSION = "0025_canonical_semantic_decision_v2"
DATE_RE = re.compile(r"20\d\d-\d\d-\d\d")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_csv(path: Path, rows: list[Mapping[str, Any]], fields: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

def path_date(path: Path) -> date | None:
    for value in DATE_RE.findall(str(path)):
        try:
            parsed = date.fromisoformat(value)
        except ValueError:
            continue
        if DATA_START <= parsed <= DATA_END:
            return parsed
    return None

def input_files(root: Path = Path("/kaggle/input")) -> list[Path]:
    return sorted(path for path in root.rglob("*") if path.is_file() and path_date(path) is not None)

def payloads(path: Path):
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as bundle:
            for member in bundle.namelist():
                if not member.lower().endswith(".json"):
                    continue
                try:
                    value = json.loads(bundle.read(member))
                except Exception:
                    continue
                if isinstance(value, dict):
                    yield f"{path}!{member}", value
        return
    try:
        opener = gzip.open if path.name.lower().endswith(".gz") else open
        with opener(path, "rt", encoding="utf-8-sig") as handle:
            value = json.load(handle)
        if isinstance(value, dict):
            yield str(path), value
        elif isinstance(value, list):
            for index, item in enumerate(value):
                if isinstance(item, dict):
                    yield f"{path}#{index}", item
    except Exception:
        return

def first(value: Mapping[str, Any], *keys: str, default: Any = None) -> Any:
    for key in keys:
        if key in value and value[key] is not None:
            return value[key]
    return default

def as_list(value: Any) -> list[Any]:
    return value if isinstance(value, list) else []

def as_int(value: Any, default: int = 0) -> int:
    try:
        return int(value)
    except (TypeError, ValueError):
        return default

def episode_id(payload: Mapping[str, Any], fallback: str) -> str:
    info = payload.get("info")
    if isinstance(info, Mapping):
        value = first(info, "EpisodeId", "episode_id", "episodeId")
        if value is not None:
            return str(value)
    return str(first(payload, "episode_id", "episodeId", "id", default=fallback))

def winner(payload: Mapping[str, Any]) -> int | None:
    rewards = first(payload, "rewards", "reward", "scores", default=[])
    if not isinstance(rewards, list):
        return None
    indexes = [i for i, value in enumerate(rewards) if isinstance(value, (int, float)) and not isinstance(value, bool) and value > 0]
    return indexes[0] if len(indexes) == 1 else None

def deck_hash(cards: Any) -> str | None:
    ids = []
    for value in as_list(cards):
        if isinstance(value, Mapping):
            value = first(value, "id", "cardId")
        try:
            ids.append(int(value))
        except (TypeError, ValueError):
            return None
    if len(ids) != 60:
        return None
    return hashlib.sha256(",".join(map(str, sorted(ids))).encode("ascii")).hexdigest()

def frames(payload: Mapping[str, Any]):
    values = first(payload, "steps", "frames", "trajectory", "observations", default=[])
    for index, frame in enumerate(as_list(values)):
        if not isinstance(frame, Mapping):
            continue
        observation = first(frame, "observation", "obs", "state", default=frame)
        if not isinstance(observation, Mapping) or not isinstance(observation.get("select"), Mapping):
            continue
        current = observation.get("current") if isinstance(observation.get("current"), Mapping) else {}
        actor = as_int(first(frame, "playerIndex", "player", "actor", default=current.get("yourIndex", 0)))
        action = first(frame, "action", "selected", "ordered_action", "selection", default=[])
        yield index, actor, observation, action

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/ptcg_0027_dataset")
OUT.mkdir(parents=True, exist_ok=True)
source = next(INPUT.rglob("canonical_records.jsonl.gz"), None)
if source is None:
    raise FileNotFoundError("attach notebook 03 output containing canonical_records.jsonl.gz")
ACTOR_KEYS = {"global_cat", "global_num", "card_cat", "card_num", "card_parent", "resource_cat", "resource_num", "event_cat", "event_num", "option_cat", "option_num", "option_state", "option_source", "option_target", "option_skill_id", "option_skill_role", "option_skill_parent", "option_effect_id", "option_effect_role", "option_effect_parent", "min_count", "max_count"}
cat_widths = {"global_cat": 11, "card_cat": 5, "resource_cat": 4, "event_cat": 8, "option_cat": 14}
num_widths = {"global_num": 17, "card_num": 7, "resource_num": 15, "event_num": 4, "option_num": 16, "option_state": 16}
rows = {"train": [], "validation": []}
episodes = {}
with gzip.open(source, "rt", encoding="utf-8") as handle:
    for line in handle:
        record = json.loads(line)
        actor, target = record["actor"], record["target"]
        if set(actor) != ACTOR_KEYS or any(key in actor for key in ("legacy", "action", "source_id", "source_team_name")):
            raise ValueError("actor leakage or key mismatch")
        for key, width in {**cat_widths, **num_widths}.items():
            values = actor[key]
            if key in ("global_cat", "global_num"):
                if len(values) != width:
                    raise ValueError(f"{key} width mismatch")
            elif values and len(values[0]) != width:
                raise ValueError(f"{key} width mismatch")
        option_count = len(actor["option_cat"])
        action = [int(value) for value in target["ordered_action"]]
        if not option_count or len(set(action)) != len(action) or any(value < 0 or value >= option_count for value in action):
            raise ValueError("illegal ordered action")
        if not int(actor["min_count"]) <= len(action) <= int(actor["max_count"]):
            raise ValueError("min/max contract violated")
        identity = record["audit"]["identity"]
        episode = "::".join(str(identity.get(name, "")) for name in ("date", "episode_id", "player_index"))
        split = record["audit"].get("split", "train")
        if split not in rows:
            split = "validation" if int(hashlib.sha256(episode.encode()).hexdigest()[:8], 16) % 10 == 0 else "train"
        if episode in episodes and episodes[episode] != split:
            raise ValueError(f"episode split leakage: {episode}")
        episodes[episode] = split
        rows[split].append(record)

def shards(split: str, values: list[dict[str, Any]]) -> list[dict[str, Any]]:
    result = []
    for start in range(0, len(values), 2048):
        chunk = values[start:start + 2048]
        path = OUT / f"{split}-{start // 2048:05d}.jsonl.gz"
        with gzip.open(path, "wt", encoding="utf-8") as handle:
            for row in chunk:
                handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n")
        result.append({"path": path.name, "count": len(chunk), "bytes": path.stat().st_size, "sha256": sha256_file(path)})
    return result

manifest = {"schema_version": SCHEMA_VERSION, "status": "complete", "split_counts": {key: len(value) for key, value in rows.items()}, "episode_split_counts": {key: len({"::".join(str(row["audit"]["identity"].get(name, "")) for name in ("date", "episode_id", "player_index")) for row in value}) for key, value in rows.items()}, "records_per_shard": 2048, "shards": {key: shards(key, value) for key, value in rows.items()}, "actor_forward_excludes": ["legacy", "ordered_action", "source_id", "source_team_name", "source_payload_sha256"], "source_identity_actor_visible": False, "initialized_from_0025": {"checkpoint": "best_greedy_exact.pt", "sha256": "adc4eaeca1e62a28bbd762e8e513212c94941044bbc85673f02bbeadc1865aa3", "bytes": 129028675}}
write_json(OUT / "manifest.json", manifest)
write_json(OUT / "dataset_reference.json", {"manifest_sha256": sha256_file(OUT / "manifest.json"), "manifest": manifest})
print(json.dumps({"train": len(rows["train"]), "validation": len(rows["validation"]), "output": str(OUT)}, indent=2))

